Two of the strongest action streams on this board, a hand-written public
script and the private engine of the team holding rank 1 in mid August, execute their farms with
the same routing efficiency to two decimal places, while a rules-driven
dispatcher I measured runs 82 % worse against the same yardstick. Numbers
like that are not a coincidence to admire, they are a problem statement: what
optimization problem is a Kaggriculture day, formally, and what does its
structure say about how it should be solved?

This notebook models the day as a **Team Orienteering Problem with Time
Windows** with three structures the engine imposes on top, states the model's
assumptions against engine source (1.32.7), and validates its one central
prediction with a bound that can be checked. Everything computes from
embedded data; the replay corpora are public.

## 1. Why the day, and not the episode, is the instance

Four engine facts, each from the 1.32.7 source, make the in-game day the
natural planning unit:

1. every unit returns to spawn at the nightly rollover, and **the hired crew
   dissolves**: hiring is per-day, so the fleet itself is daily;
2. `watered_today`, `fed_today` and `cared_today` reset at the rollover, so
   the day's task set regenerates each morning;
3. weeds spawn at the rollover, not during the day, so **the farm layer is
   deterministic intra-day given your own actions**: everything a router
   needs is known at hour 0;
4. prices and the opponent move intra-day, but they touch the *market*
   channel, not the geometry of chores.

So a 720-turn episode decomposes into 30 near-independent daily routing
instances chained through a narrow synchronisation bottleneck. The nightly
reset is literally *return to depot plus fleet dissolution*.

## 2. The model

**Instance (one day $d$).** A $10\times10$ grid, a depot $S$ (the four
shed-access tiles), a horizon of 24 unit-hours, and a fleet: the farmer plus
$h$ hired hands, where the $n$-th hire of the day costs $\mu F_n$ with $F_n$
the Fibonacci sequence, so fleet size is a *decision with convex marginal
cost*, not a given. Each unit moves one tile per hour (Manhattan) or executes
one action on its tile; **travel costs no money, only foregone work**, which
is the correct cost model when labour is the binding resource.

**Jobs.** The day's job set $J_d$ is known at hour 0 (fact 3 above). Each job
$j$ has a tile $x_j$, a window $[a_j, b_j] \subseteq [0,23]$ and a prize
$p_j \ge 0$, and jobs are **skippable**: the objective is collected value,
which places the problem in the orienteering family (vehicle routing *with
profits*), not the classical vehicle routing problem with time windows
(VRPTW) where all customers must be served. The acronyms introduced here,
TOPTW, VRPTW, VRPSTW for the soft-window variant, PDPTW for
pickup-and-delivery, and MST for the spanning-tree bound, are used
throughout the rest of the text. Three structures sit
on top:

* **Watering is a constraint and a prize at once.** A plant dies after two
  consecutive unwatered days, so a tile at `consecutive_unwatered = 1` is a
  *hard deadline tonight* whose prize is the discounted value of the plant's
  remaining production, while a tile watered yesterday is legally skippable.
  And inside the growth window of a non-ongoing crop each water adds $+1$
  yield ($+2$ if fertilized), engine-exact, so in-window waters carry a
  direct production prize on top of survival.
* **Harvest windows are hard-open, soft-close.** The engine silently refuses
  harvest before `first_yield_day` (hard $a_j$), and after
  `max_lifespan_step` the standing yield decays one unit every two turns: a
  *soft* time window with linear lateness penalty, the shape the routing
  literature calls VRPSTW.
* **Feeding is a pickup-and-delivery strand.** FEED consumes one wheat from
  the executing unit's own inventory, so each feed is a pickup (shed) to
  delivery (animal tile) pair with pairing and precedence, the defining
  constraints of PDPTW. An animal escapes after two consecutive unfed days,
  and the care production bonus is consumed only on fed production days, so
  feed prizes chain across days.

**Objective.**

$$\max_{\;h,\;\text{routes}}\;\;\sum_{j \in \text{served}} p_j\;-\;\sum_{j
\in \text{late}} \lambda_j(t_j)\;-\;\sum_{n=1}^{h} \mu F_n$$

subject to windows, feed pairing and precedence, one action per unit-hour,
and unit-speed movement. To my knowledge the combination, orienteering with
Fibonacci fleet sizing and a PDPTW strand, has no named form in the
literature, which makes it more interesting rather than less: the nearest
classical anchors are the orienteering problem (Tsiligirides 1984; Golden,
Levy and Vohra 1987) and its team form TOP/TOPTW (Chao, Golden and Wasil
1996; survey in Vansteenwegen et al. 2011 and, current to late 2025, in
arXiv:2512.16865) for the skippable prizes, VRP with soft time windows for
harvest lateness (Taillard et al. 1997), pickup-and-delivery for feed
(Savelsbergh and Sol 1995), and fleet-size-and-mix VRP for the hire curve
(Golden et al. 1984). Full entries in the references section.

In [1]:
# engine-exact parameters, read from kaggle_environments 1.32.7 source
CROPS = {  # first_yield_day, max_yield_day, max_yield, ongoing
 "WHEAT":      dict(open=2,  close=4,  cap=6, ongoing=False),
 "CARROT":     dict(open=2,  close=3,  cap=4, ongoing=False),
 "TOMATO":     dict(open=8,  close=8,  cap=4, ongoing=True),
 "STRAWBERRY": dict(open=10, close=10, cap=4, ongoing=True),
 "MELON":      dict(open=10, close=12, cap=6, ongoing=False)}
ANIMALS = {  # structure, first yield, interval, product
 "GOOSE": dict(home="COOP",    open=4, every=1, product="EGG"),
 "COW":   dict(home="PASTURE", open=8, every=2, product="MILK"),
 "SHEEP": dict(home="PASTURE", open=6, every=3, product="WOOL")}
RULES = dict(unwatered_death=2, unfed_escape=2, harvest_decay="1 unit / 2 turns",
             water_bonus="+1 (+2 fertilized) inside [ (close+1)//2, close ]")
print("crop windows (in-game days from planting):")
for k, v in CROPS.items():
    print(f"  {k:<11} open {v['open']:>2}  close {v['close']:>2}  cap {v['cap']}  ongoing {v['ongoing']}")
print("\nanimal production:", {k: f"{v['product']} from day {v['open']} every {v['every']}" for k, v in ANIMALS.items()})
print("hard rules:", RULES)

## 3. What the model assumes, and what each relaxation costs

Stating the model is cheap; stating what was thrown away to state it is
where the honesty lives. Five assumptions, each with its implication:

1. **Market separability.** The model treats routing as independent of
   prices and the opponent. True for geometry, false at two coupling points
   I have measured elsewhere: goods must reach the shed *before* the sell
   turns that finance the plan's purchases (a late delivery de-funds
   construction), and cash-gated buys shift the job set between days. So
   the day instances are not fully independent; value flows across days
   through cash. *Implication:* the routing model governs the unit channel
   only, and delivery deadlines must enter as hard windows derived from the
   sell schedule, which is how I use it.
2. **Prize additivity.** The objective sums per-job prizes, but real prizes
   chain: feeding today preserves every future fertilizer night, the care
   bonus pays only on fed production days, a water's value depends on
   future harvests and on scarcity-curved prices. *Implication:* the $p_j$
   are marginal values of a season-level dynamic program approximated by
   constants; the approximation is worst where coupling is strongest, the
   herd, and any solver tuned on constant prizes inherits that bias.
3. **A fixed fleet.** Standard TOPTW fixes the fleet; here hires arrive
   *during* the morning by the financing schedule, so the fleet is
   time-varying within the day. *Implication:* plans must be crew-ramp
   aware or replan on crew change; I measured this as a necessity, not a
   nicety.
4. **A memoryless day.** Skipping a water today creates a hard deadline
   tomorrow (`consecutive_unwatered` 1), so daily instances share state and
   the season is really a 30-stage dynamic program whose stage rewards the
   daily prizes approximate. *Implication:* solving days myopically is a
   rolling-horizon approximation, fine while inter-day coupling is weak and
   wrong at the boundaries, the liquidation days, which need their own
   objective.
5. **Free coexistence.** Units neither collide nor congest, engine-exact,
   so no capacity constraint on tiles. This one costs nothing; it is in the
   model's favor.

## 4. The prediction the model makes, and the instrument that checks it

If the field's scripts are *solutions of this daily problem, solved offline
and cached as choreography*, then their executed travel should sit at a
consistent multiple of a routing lower bound, and any dispatcher of the same
quality should land on the same multiple whether or not it is a script.

The bound: the union of one day's unit walks, all starting at the depot,
must connect every tile where work executed, so total daily travel is at
least the weight of a **minimum spanning tree (MST) over the served tiles
plus the depot**, the classical relaxation of tour length (Held and Karp
1970). This bound is *deliberately loose*: it ignores time windows, revisit
multiplicity (a tile watered at hour 3 and harvested at hour 20 forces two
arrivals) and the feed round-trips. It therefore cannot certify optimality.
What it can do is compare profiles fairly, because every profile faces the
same looseness.

In [2]:
# The dataset for every figure below, embedded once. profiles: season
# travel, its MST bound, their ratio, the windows-relaxed re-solve and the
# slack against it, for the public script, the then-leader's engine and my
# rules dispatcher. instance: one real served-tile day. daily: per-day
# (day, travel, mst) triples per profile. fib: the hire cost sequence.
import json
import matplotlib.pyplot as plt
D = json.loads(r'''{"profiles":{"script":{"travel":3625,"mst":1415,"ratio":2.56,"solved":3000,"slack":1.21},"king":{"travel":3695,"mst":1446,"ratio":2.56,"solved":2990,"slack":1.24},"rules":{"travel":4317,"mst":927,"ratio":4.66,"solved":1924,"slack":2.24}},"instance":{"day":12,"tiles":[[0,0],[0,2],[0,4],[1,1],[1,3],[2,0],[2,1],[2,2],[2,3],[2,4],[3,0],[3,1],[3,2],[3,3],[3,4],[3,5],[3,8],[4,0],[4,1],[4,2],[4,3],[4,4],[4,5],[4,6],[4,8],[5,1],[5,2],[5,3],[5,4],[5,5],[6,0],[6,2],[6,3],[6,4],[7,1],[7,3],[7,4],[8,0],[8,2],[8,4],[9,1],[9,3],[9,4]],"travel":154,"work":91,"mst":51,"episode":"93479646"},"fib":[1,1,2,3,5,8,13,21,34,55,89,144,233,377],"daily":{"script":[[0,52,22],[1,10,3],[2,45,23],[3,48,20],[4,51,21],[5,44,19],[6,69,26],[7,104,29],[8,152,43],[9,120,40],[10,164,43],[11,156,60],[12,154,55],[13,122,63],[14,116,48],[15,171,68],[16,133,51],[17,167,62],[18,175,58],[19,156,67],[20,153,65],[21,169,65],[22,142,70],[23,165,59],[24,142,64],[25,135,59],[26,132,68],[27,144,60],[28,113,58],[29,121,26]],"king":[[0,52,22],[1,10,3],[2,45,23],[3,48,20],[4,51,21],[5,44,19],[6,69,25],[7,103,30],[8,147,43],[9,131,40],[10,164,43],[11,156,60],[12,154,51],[13,124,62],[14,117,51],[15,178,63],[16,126,58],[17,166,64],[18,155,64],[19,147,67],[20,152,66],[21,179,67],[22,149,69],[23,165,68],[24,154,59],[25,151,63],[26,148,65],[27,150,67],[28,123,55],[29,137,38]],"rules":[[0,58,22],[1,10,3],[2,63,23],[3,69,23],[4,93,23],[5,83,18],[6,68,21],[7,91,25],[8,81,23],[9,93,23],[10,186,23],[11,191,42],[12,154,44],[13,179,44],[14,130,31],[15,194,39],[16,146,33],[17,195,41],[18,192,39],[19,203,41],[20,195,31],[21,194,35],[22,191,39],[23,197,35],[24,183,35],[25,187,33],[26,188,34],[27,183,41],[28,181,32],[29,139,31]]}}''')
P = D["profiles"]
print(f"{'profile':<38}{'travel':>8}{'MST bound':>11}{'ratio':>7}")
rows = [("public script 10C4S (one run vs IDLE)", "script"),
        ("kawashigi rank 1 (one real episode)", "king"),
        ("my rules engine v8.8 (one run vs IDLE)", "rules")]
for label, k in rows:
    p = P[k]
    print(f"{label:<38}{p['travel']:>8,}{p['mst']:>11,}{p['ratio']:>7}")

In [3]:
COL = {"script": "#0F172A", "king": "#B45309", "rules": "#0E7490"}
LAB = {"script": "public script", "king": "kawashigi", "rules": "rules engine v8.8"}
fig, ax = plt.subplots(figsize=(9, 3.6))
for k in ("script", "king", "rules"):
    pts = [(d, tr / m) for d, tr, m in D["daily"][k] if m >= 8]
    xs, ys = zip(*pts)
    ax.plot(xs, ys, color=COL[k], lw=2, label=LAB[k])
    ax.annotate(LAB[k], (xs[-1], ys[-1]), xytext=(4, 0),
                textcoords="offset points", color=COL[k], fontsize=9,
                va="center")
ax.set_xlabel("in-game day")
ax.set_ylabel("travel paid / MST bound")
ax.set_xlim(0, 34)
ax.grid(alpha=0.25)
ax.legend(loc="upper left", fontsize=9, frameon=False)
ax.set_title("routing multiple by day: the gap is season-long, not one phase",
             fontsize=10)
plt.tight_layout(); plt.show()

The day-level view adds one fact the season totals hide: the leaders'
multiple is *stable* across the whole season while the rules engine runs
above them nearly everywhere, so the inefficiency is a property of the
dispatcher, not of one game phase (the construction days, where few tiles
are served and the bound is tiny, are filtered at MST < 8 to keep the ratio
meaningful).

## 5. Putting the plateau claim to the test

The shared 2.56 invites the phrase "the scripts are near-optimal cached
solutions". That phrase hides two different claims, a *structural floor*
(windows, revisits and depot trips force 2.56, nobody can do better) and a
*converged plateau* (better exists, the population just settled). They are
distinguishable, so I test them: for each profile's OWN days, same served
tiles, same crew size, I run a cheap solver (balanced nearest-insertion plus
a 2-opt pass per unit, hours relaxed) and compare its travel to the travel
actually paid. The solver is insertion plus 2-opt (Solomon 1987; Croes
1958), an *upper* bound on the optimum, so whatever it saves is slack that
exists beyond connectivity; and because it relaxes hour windows, part of
that saving is legitimately unavailable, which bounds the truth from both
sides. One caveat the state of the art imposes: this solver is far from
modern strength (hybrid genetic search closes routing gaps my two moves
cannot, Vidal 2022), so its savings are a LOWER estimate of the true slack,
a point the limits section returns to.

In [4]:
P = D["profiles"]
print(f"{'profile':<26}{'paid':>7}{'solver upper':>14}{'paid/solver':>13}{'solver/MST':>12}")
for label, k in (("public script", "script"), ("kawashigi rank 1", "king"),
                 ("my rules engine v8.8", "rules")):
    p = P[k]
    print(f"{label:<26}{p['travel']:>7,}{p['solved']:>14,}{p['slack']:>13}"
          f"{p['solved']/p['mst']:>12.2f}")

In [5]:
P = D["profiles"]
keys = ["script", "king", "rules"]
labs = ["public script", "kawashigi", "rules engine v8.8"]
paid = [P[k]["travel"] for k in keys]
solv = [P[k]["solved"] for k in keys]
mst = [P[k]["mst"] for k in keys]
x = range(3)
fig, ax = plt.subplots(figsize=(8, 3.8))
b1 = ax.bar([i - 0.2 for i in x], paid, width=0.36, color="#0F172A",
            label="travel paid")
b2 = ax.bar([i + 0.2 for i in x], solv, width=0.36, color="#0E7490",
            label="cheap solver on the same days (upper bound)")
for i in x:
    ax.plot([i - 0.45, i + 0.45], [mst[i]] * 2, color="#B45309", lw=2.2)
    ax.annotate(f"{paid[i]:,}", (i - 0.2, paid[i]), ha="center",
                va="bottom", fontsize=9)
    ax.annotate(f"{solv[i]:,}", (i + 0.2, solv[i]), ha="center",
                va="bottom", fontsize=9)
ax.plot([], [], color="#B45309", lw=2.2, label="MST lower bound")
ax.set_xticks(list(x)); ax.set_xticklabels(labs)
ax.set_ylabel("unit-hours of travel, season total")
ax.grid(alpha=0.25, axis="y")
ax.legend(loc="upper right", fontsize=9, frameon=False)
ax.set_title("the test at a glance: the optimum lives between the amber line "
             "and the teal bar", fontsize=10)
plt.tight_layout(); plt.show()

## 6. Reading the three layers, with the equilibrium caveat stated first

The script and the strongest private engine on the board sit at **the same
ratio to two decimals**, and my rules dispatcher sits 82 % above them on its
own, smaller, instance set (the MST column shows the instances differ in
size; the ratio normalises within profile).

The test decomposes the 2.56 into layers. Multi-visit structure and depot
trips, with hours relaxed, already cost about 2.1 times the MST (the
solver/MST column), so most of the leaders' multiple is *structure, not
waste*. What remains is about 20 %: the script pays 1.21 times its own
relaxed-solver route, the king 1.24, and that residue is the sum of two
things this instrument cannot separate, travel that hour windows genuinely
force, and local-optimum slack the field never crossed. So the floor
hypothesis is REFUTED in its strong form, better-than-executed routes exist
for their own workloads, while near-optimality *under windows* survives
inside a bounded 20 % corridor. Three readings of why the two strongest
profiles land on the same number remain, and the field's own dynamics rank
them:

1. **A shared local optimum.** The board evolves by copying and local
   mutation, and that process converges to local optima of the day problem,
   not global ones. A common execution plateau is the *expected equilibrium
   condition* of such a population, which makes the identical ratio evidence
   of convergence, not of optimality.
2. **Common descent.** The rank-1 engine and the public script are not
   independent solvers; the top of this board shares ancestry, so a shared
   plateau may simply be inherited.
3. **A structural floor.** Windows, revisits and feed round-trips impose
   travel the MST does not count; some of the 2.56 is that floor, the same
   for everyone.

The three are not exclusive and the bound cannot separate them. What the
test separates cleanly is MY OWN slack: v8.8 pays **2.24 times** the cheap
solver's route for its own, smaller, workload, against the leaders' 1.21
and 1.24. That is roughly 2,400 unit-hours a season, larger than the entire
work-volume gap (~2,050 vs ~2,700 useful actions) between my engine and the
leaders, so the model's actionable quantity is quantified and it is mine to
collect, not theirs.

In [6]:
wi = D["instance"]
tiles = [tuple(t) for t in wi["tiles"]]
depot = (4, 4)
pts = tiles + [depot]
# recompute the MST edges for the picture (Prim)
in_tree, out, edges = {pts[-1]}, set(pts[:-1]), []
while out:
    best = None
    for p in out:
        for q in in_tree:
            dd = abs(p[0]-q[0]) + abs(p[1]-q[1])
            if best is None or dd < best[0]:
                best = (dd, p, q)
    edges.append((best[1], best[2])); in_tree.add(best[1]); out.discard(best[1])
fig, ax = plt.subplots(figsize=(5.4, 5.4))
for (a, b) in edges:
    ax.plot([a[0], b[0]], [a[1], b[1]], color="#B45309", lw=1.4, zorder=1)
xs, ys = zip(*tiles)
ax.scatter(xs, ys, s=170, c="#DCEEE1", edgecolors="#0F172A", zorder=2,
           label=f"served tiles ({len(tiles)})")
ax.scatter(*depot, marker="s", s=200, c="#0F172A", zorder=3, label="depot (shed)")
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlim(-0.6, 9.6); ax.set_ylim(9.6, -0.6)
ax.grid(alpha=0.25); ax.legend(loc="lower right", fontsize=9)
ax.set_title(f"one real day of rank 1 (in-game day {wi['day']}): "
             f"MST bound {wi['mst']} vs travel paid {wi['travel']}", fontsize=10)
plt.tight_layout(); plt.show()
print(f"episode {wi['episode']}: {wi['work']} work actions on {len(tiles)} tiles,")
print(f"travel {wi['travel']} against a connectivity bound of {wi['mst']}")

## 7. The fleet term

The $n$-th hire of a day costs $\mu F_n$: 1, 1, 2, 3, 5, 8, 13, 21, 34
units of $\mu$, and the crew dissolves at night. Two consequences the model
makes explicit. First, labour is nearly free at small $n$, a full working
crew costs less than one day of one animal's output, which is why every
strong profile runs 10-12 hands mid-season. Second, the convex curve prices
the *marginal* hand against the marginal uncollected prize of the day plan,
so fleet size is properly part of the optimization, not a constant: a day
whose job prizes are exhausted by 8 units should not pay $\mu F_9$ for a
ninth. When this was first written no measured profile exploited the margin
adaptively. A wider census since then sharpens that: over hundreds of real
episodes the current rank 1 hires **23** units across days 0 to 5 where my
own agent hires 36, and carries about **330** idle-with-work unit-turns a
season against my 395, so the leader is not sizing adaptively either, but
his fixed schedule sits measurably closer to the margin this section
derives. The gap between a tuned constant and the adaptive rule remains
unclaimed by anyone I can measure.

In [7]:
fib = D["fib"][:12]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(1, len(fib) + 1), fib, color="#0E7490")
for n, f in enumerate(fib, 1):
    ax.annotate(str(f), (n, f), ha="center", va="bottom", fontsize=9)
ax.set_xticks(range(1, len(fib) + 1))
ax.set_xlabel("n-th hand hired that day")
ax.set_ylabel("marginal cost, units of the hire multiplier")
ax.grid(alpha=0.25, axis="y")
ax.set_title("the fleet term: convex by Fibonacci, near-free until it is not",
             fontsize=10)
plt.tight_layout(); plt.show()

## 8. Algorithmic applications under this framework, with the balance stated

What the framework opens, each with an honest cost-benefit line:

| application | cost | benefit | assessment |
|---|---|---|---|
| **day-plan selection by rollout** (MPC by rollout on combinatorial structure: Bertsekas, Tsitsiklis and Wu 1997; Bertsekas 2020) | medium: a forward model plus a plan generator | quantified: the 2.24x-vs-1.2x corridor is ~2,400 unit-hours a season, larger than the leaders' entire work-volume edge | chosen here; the success criterion is landing under ~1.2x your own relaxed solver |
| **crew sizing** (the Fibonacci term as a live decision) | small, once a plan values its marginal job | bounded by the convex curve; no measured profile exploits it, so the edge is real but unproven | after rollout exists, priced against its plan values |
| **harvest-to-schedule alignment** (due-date scheduling of harvests against the sell turns) | small; it is a due-date rule | measured: one missed melon window cost a fifth of a season's bank in an earlier build | already worth it as a hard window in any solver |
| **offline route synthesis** (solve the 30 instances offline, emit a fixed script) | medium-high | would generate competitive fixed routes *without copying anyone*; the board's own economy proves scripts of this quality hold rating | legitimate and unexplored; inherits every fragility of open-loop play |
| **an execution-quality instrument** (paid-over-own-solver ratio as a gate for any candidate) | tiny; the solver runs offline on replays | grades execution of ANY agent cheaply, no games spent | adopt regardless of the rest |
| **full season DP** (stage-coupled prizes, liquidation boundary) | high | unknown; assumption 2 says the constant-prize bias concentrates in the herd, so the DP's edge may be narrow | parked until rollout plateaus |

* **The equilibrium framing sets the target correctly.** The goal is not
  "beat the MST bound", it is *cross the corridor*: the leaders sit within
  about 20 % of a windows-relaxed solver on their own days, so any
  dispatcher that lands nearer than 1.2 times its relaxed solution is, by
  this instrument, executing better than the converged field.
* **A genome connection.** In an earlier notebook I showed each replay
  carries a 30-band signature, one band per day. In this frame a band is a
  *hashed solution of that day's routing instance* (its anchor-window plan),
  so a cached script and a live solver differ exactly as a memoised answer
  differs from a re-computation, which is why the deepest adaptive agent on
  the board is precisely the one whose bands agree least with themselves.

## 9. Limits and scope of the model

Seven limits bound what this framework can claim, each with its
consequence:

1. **The decomposition is leaky.** Watering flags, the herd and cash couple
   consecutive days, so the daily instance is a stage of a season-level
   process, not an isolated problem. The coupling enters through prizes and
   through delivery windows (assumptions 1 and 2), not through route
   feasibility, so the model governs routing; it cannot, by itself, set the
   prizes.
2. **Geometry differs across farms.** A smaller farm has a smaller bound,
   so cross-profile ratio comparisons are not rankings. Every comparative
   number here is within-profile, paid against that profile's own bound or
   its own re-solved routes; the only cross-profile observation is that two
   strong profiles landed on the same multiple.
3. **Routing quality has not been converted to bank end to end.** Each link
   of the chain, travel to work-hours to in-window waters to yield, is an
   engine rule or a measured fact, but the conversion rate is not yet
   established by a controlled experiment. Until it is, the framework
   describes execution; it does not yet guarantee that improving execution
   moves the score.
4. **The shared multiple has three readings.** A common local optimum,
   common descent, and a structural floor all fit the 2.56, and the
   instruments here cannot separate them. Nothing downstream depends on
   which is true: the actionable quantity is a profile's distance to its
   own re-solved routes.
5. **Every corridor number is solver-relative.** The re-solving used
   insertion plus 2-opt; solvers of hybrid-genetic-search strength (Vidal
   2022) would tighten the upper bound and widen every measured gap, so
   1.21-1.24 is an upper estimate of the leaders' closeness and the 1.2x
   criterion re-anchors whenever the solver strengthens.
6. **Learned solvers are out of scope, not out of the question.** Neural
   combinatorial optimization solves routing classes of this kind (Kool,
   van Hoof and Welling 2019) and would be a legitimate offline tool; at
   30-50 jobs on a 10x10 grid, with a one-second stdlib runtime budget,
   the classical toolkit is adequate and simpler to audit.
7. **Anticipation is mostly, not entirely, unnecessary.** The stochastic
   dynamic routing literature locates value in anticipating information
   that arrives during execution (Soeffker, Ulmer and Mattfeld 2022). Here
   the farm layer is deterministic intra-day, which removes most of that
   channel; what remains dynamic, the morning crew ramp and cash-gated
   construction, is handled by replanning triggers, and if those couplings
   were stronger they would have to be promoted from triggers to
   first-class model elements.

Within those limits, the framework serves three uses: a language for the
design space, an instrument whose comparisons are bounded and
self-normalized, and a license for day-granular planning. The equation
"daily TOPTW optimality equals winning" is not among its claims; it is the
hypothesis the next experiment tests.

## Summary

A Kaggriculture day is a Team Orienteering Problem with Time Windows,
extended by a pickup-and-delivery strand (feeding), mixed hard/soft windows
(watering deadlines and prize, harvest decay), and endogenous Fibonacci
fleet sizing, an apparently unnamed combination. The season is 30 such
instances chained by a nightly return-to-depot that also dissolves the
fleet. Against a connectivity lower bound on daily travel, the public script and
the rank-1 private engine execute at an identical 2.56, and a re-solving
test on their own days splits that number into layers: about 2.1 is
multi-visit structure, and the leaders pay only 1.21-1.24 times a cheap
windows-relaxed solver, a bounded 20 % corridor of windows plus possible
local-optimum slack, which refutes the strong floor reading while leaving
the converged-plateau reading standing. A rules dispatcher I measured pays
2.24 times its own relaxed solution, and that difference, not the bound, is
the actionable quantity. The framework's practical yield is a licensed solver
shape, day-granular plan-by-rollout over a deterministic intra-day farm
model, six registered applications with their costs and benefits stated, a
reinterpretation of the replay genome, a script is a cached solution and a
live agent is an online solver, and a stated scope: every corridor number
is relative to the solver that produced it, and the conversion from routing
quality to bank, plausible and link-by-link measured, awaits its controlled
experiment.

One connection reaches outside this notebook. If a day is a routing instance,
then a recorded season is a LIBRARY of solved instances, and the strongest
agents on this board are exactly that: recorded seasons replayed as
choreography, some of them routed between several recordings by the shop
draws the seed fixes. The 2.56 shared to two decimals by a script and the
rank-1 engine is what "solved offline and cached" looks like from the travel
side, and it is the routing-theoretic reason the replay family, which looks
like it should not work, holds the top of the ladder.


## References

* Bertsekas, D. P. (2020). *Rollout, Policy Iteration, and Distributed
  Reinforcement Learning.* Athena Scientific.
* Bertsekas, D. P., Tsitsiklis, J. N., Wu, C. (1997). Rollout algorithms
  for combinatorial optimization. *Journal of Heuristics* 3.
* Chao, I-M., Golden, B. L., Wasil, E. A. (1996). The team orienteering
  problem. *European Journal of Operational Research* 88(3), 464-474.
* Croes, G. A. (1958). A method for solving traveling-salesman problems.
  *Operations Research* 6.
* Golden, B. L., Assad, A. A., Levy, L., Gheysens, F. (1984). The fleet
  size and mix vehicle routing problem. *Computers & Operations Research*
  11.
* Golden, B. L., Levy, L., Vohra, R. (1987). The orienteering problem.
  *Naval Research Logistics* 34, 307-318.
* Held, M., Karp, R. M. (1970). The traveling-salesman problem and minimum
  spanning trees. *Operations Research* 18.
* Kool, W., van Hoof, H., Welling, M. (2019). Attention, learn to solve
  routing problems! *ICLR 2019*, arXiv:1803.08475.
* Savelsbergh, M. W. P., Sol, M. (1995). The general pickup and delivery
  problem. *Transportation Science* 29.
* Shen, S., Zhou, Y., Lei, Q., Wu, Z. (2025). A survey of the orienteering
  problem: model evolution, algorithmic advances, and future directions.
  arXiv:2512.16865.
* Soeffker, N., Ulmer, M. W., Mattfeld, D. C. (2022). Stochastic dynamic
  vehicle routing in the light of prescriptive analytics: a review.
  *European Journal of Operational Research* 298(3), 801-820.
* Solomon, M. M. (1987). Algorithms for the vehicle routing and scheduling
  problems with time window constraints. *Operations Research* 35.
* Taillard, E., Badeau, P., Gendreau, M., Guertin, F., Potvin, J.-Y.
  (1997). A tabu search heuristic for the vehicle routing problem with
  soft time windows. *Transportation Science* 31, 170-186.
* Tsiligirides, T. (1984). Heuristic methods applied to orienteering.
  *Journal of the Operational Research Society* 35, 797-809.
* Vansteenwegen, P., Souffriau, W., Van Oudheusden, D. (2011). The
  orienteering problem: a survey. *European Journal of Operational
  Research* 209(1), 1-10.
* Vidal, T. (2022). Hybrid genetic search for the CVRP: open-source
  implementation and SWAP* neighborhood. *Computers & Operations Research*
  140, 105643.

Volume-and-page details are quoted only where I verified them against the
publisher's record; entries without pages are deliberate.